# 07 - Numerical vs Categorical Feature Selection

This notebook completes the assignment on identifying, separating, validating, and documenting feature types before modeling.

## Why This Matters
Feature type selection drives:
- Encoding strategy
- Scaling decisions
- Model compatibility
- Interpretability
- Leakage risk
- Training stability

Many modeling failures are preprocessing failures caused by incorrect feature typing.

## Core Concepts

### Numerical Features
Numerical features carry magnitude and meaningful arithmetic relationships (continuous or discrete counts).

### Categorical Features
Categorical features represent labels or groups where arithmetic is not meaningful.

### Critical Rule
Storage dtype does not define ML feature type.
A column stored as integer may still be categorical in meaning (for example binary flags).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_data

df = load_data(n_samples=500, random_state=42)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

## Step 1 - Inspect Before Classifying
Use structure, summary, and uniqueness checks to avoid assumptions.

In [ ]:
print("DataFrame info:")
print(df.info())

print("\nNumeric describe:")
display(df.describe(include=["number"]))

unique_counts = df.nunique().sort_values()
display(unique_counts.to_frame(name="n_unique"))

## Step 2 - Define Target First
Target must be removed from candidate feature pools immediately to prevent leakage.

In [ ]:
TARGET_COLUMN = "target"

assert TARGET_COLUMN in df.columns, f"Target column '{TARGET_COLUMN}' is missing"

X_raw = df.drop(columns=[TARGET_COLUMN]).copy()
y = df[TARGET_COLUMN].copy()

print("X_raw shape:", X_raw.shape)
print("y shape:", y.shape)
print("Target distribution:")
print(y.value_counts(normalize=True))

## Step 3 - Create Explicit Feature Groups

For this project dataset, all predictor columns are numeric sensor measurements.
The assignment still requires explicit grouping and documentation, so we define both lists intentionally.

In [ ]:
NUMERICAL_FEATURES = [
    "feature_1",
    "feature_2",
    "feature_3",
    "feature_4",
    "feature_5",
    "feature_6",
]

CATEGORICAL_FEATURES = []

EXCLUDED_COLUMNS = []

ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

print("Numerical:", NUMERICAL_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
print("Excluded:", EXCLUDED_COLUMNS)

## Step 4 - Validate Feature Configuration
Checks for overlap, missing columns, and boundary violations.

In [ ]:
def validate_feature_configuration(dataframe, target_col, num_cols, cat_cols, excluded_cols):
    all_cols = num_cols + cat_cols

    assert target_col not in all_cols, "Leakage: target present in feature list"
    assert set(num_cols).isdisjoint(set(cat_cols)), "Overlap between numerical and categorical lists"
    assert set(excluded_cols).isdisjoint(set(all_cols)), "Excluded columns found in feature list"

    missing = sorted(set(all_cols) - set(dataframe.columns))
    assert not missing, f"Configured features missing in data: {missing}"

    unknown_excluded = sorted(set(excluded_cols) - set(dataframe.columns))
    if unknown_excluded:
        print("Warning: excluded columns not present in dataframe:", unknown_excluded)

    print("Feature configuration validation passed.")

validate_feature_configuration(
    dataframe=df,
    target_col=TARGET_COLUMN,
    num_cols=NUMERICAL_FEATURES,
    cat_cols=CATEGORICAL_FEATURES,
    excluded_cols=EXCLUDED_COLUMNS,
)

## Step 5 - Build Typed Feature Matrices
This structure is ready for preprocessing pipelines (scale numeric, encode categorical).

In [ ]:
X_num = df[NUMERICAL_FEATURES].copy()
X_cat = df[CATEGORICAL_FEATURES].copy() if CATEGORICAL_FEATURES else pd.DataFrame(index=df.index)
X = df[ALL_FEATURES].copy()

print("X_num shape:", X_num.shape)
print("X_cat shape:", X_cat.shape)
print("X shape:", X.shape)

## Handling Edge Cases (Reference)

- Binary flags (0/1): often categorical in meaning, numeric in storage
- High-cardinality categories: avoid blind one-hot encoding
- Date/time columns: derive meaningful features (day, month, recency) before modeling
- Encoded labels: do not assume integer encoding implies continuous magnitude

## Validation Checklist

Before modeling, confirm:
- Target is explicitly defined
- Numerical and categorical feature lists are explicit
- No target-feature overlap
- No ID/post-outcome columns included
- Each feature is available at prediction time
- High-cardinality features have an intentional strategy
- Grouping decisions are documented

## Closing Reflection
Feature typing is conceptual, not mechanical.

Every feature should answer:
1. Is this available at prediction time?
2. Does the assigned type match how the model should interpret it?

Correct feature structure leads to stable preprocessing and honest evaluation.

## Bonus Content

- Scikit-learn ColumnTransformer Guide: https://scikit-learn.org/stable/modules/compose.html#columntransformer-for-heterogeneous-data
- Google ML Guide - Data Preparation: https://developers.google.com/machine-learning/crash-course/data-prep